<a href="https://colab.research.google.com/github/ncsu-geoforall-lab/GIS582-assignments/blob/main/7AB%20-%20Flow%20Modeling/7B_Spring26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7B: Erosion Modeling

**Course:** [GIS 582 - Geospatial Modeling and Analysis](https://ncsu-geoforall-lab.github.io/geospatial-modeling-course/grass/hydrology_erosion.html)  
**Institution:** [NC State University, Center for Geospatial Analytics](https://cnr.ncsu.edu/geospatial/)
**Instructors:** Helena Mitasova, Corey White, and team

## Resources

- [GRASS overview and manual](https://grass.osgeo.org/grass84/manuals/index.html)
- [Brief theoretical background, equations, units](../resources/erosion_notes.pdf)
- [C-factor values](data/cfactor.txt)
- [K-factor](../resources/kfactor.html)
- [R-factor](https://www.ars.usda.gov/ARSUserFiles/64080530/RUSLE/AH_703.pdf) (page 47)

## Data

Text files with recode, color, and label rules:
- [Land use category description (lu_labels.txt)](data/lu_labels.txt)
- [C-factor recode table (cfac_rules.txt)](data/cfac_rules.txt)
- [C-factor color table (cfac_color.txt)](data/cfac_color.txt)
- [Soil loss color table (soilloss_color.txt)](data/soilloss_color.txt)
- [Erosion/deposition color table (erdep_color.txt)](data/erdep_color.txt)
- [Erosion/deposition classes (erdep_class.txt)](data/erdep_class.txt)
- [Erosion/deposition class labels (erdep_label.txt)](data/erdep_label.txt)

## Learning Objectives

In this tutorial, you will learn how to:
- Compute topographic potential (LS factor) for soil erosion
- Compute soil detachment using USLE3D
- Compute net erosion/deposition maps using the USPED model

## Tutorial Outline

* Part 1: Environment Setup
* Part 2: Compute soil detachment using USLE3D
* Part 3: Compute soil detachment for spatially variable land cover
* Part 4: Compute net erosion/deposition using USPED
* Part 5: Optional - Sediment transport and eroded DEM

---
## Part 1: Environment Setup

### Install GRASS

**Important:** This setup takes 3-5 minutes. You'll need to run it each time you start a new Colab session.

In [ ]:
!add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
!apt update
!apt-get install -y grass-core grass-dev

Check that GRASS is installed and available:

In [ ]:
!grass --version

Check which Python version is running:

In [ ]:
import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import GRASS Python packages

We need to locate GRASS Python packages using `grass --config python_path` and add them to `sys.path`.

In [ ]:
import subprocess
import os
from pathlib import Path

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj

### Download North Carolina Sample Dataset

In [ ]:
!grass --tmp-project XY --exec g.download.project url=https://grass.osgeo.org/sampledata/north_carolina/nc_spm_08_grass7.tar.gz path=/content

Download the text files with recode rules, color rules and labels:

In [ ]:
base_url = "http://fatra.cnr.ncsu.edu/geospatial-modeling-course/data"
files = [
    "cfac_rules.txt",
    "cfac_color.txt",
    "soilloss_color.txt",
    "erdep_color.txt",
    "erdep_class.txt",
    "erdep_label.txt",
    "lu_labels.txt",
]
for f in files:
    !curl -L -o {f} {base_url}/{f}

### Initialize GRASS Session

In [ ]:
# Start GRASS session
grassdata = "/content"
project = "nc_spm_08_grass7"
mapset = "user1"  # Create a new mapset for our work

# Start GRASS Session
session = gj.init(Path(project, mapset))
print("GRASS session started.")

---
## Part 2: Compute soil detachment using USLE3D

Equations and units used in this assignment are explained in this [Brief theoretical background for GIS-based erosion modeling with RUSLE3D and USPED](https://ncsu-geoforall-lab.github.io/geospatial-modeling-course/resources/erosion_notes.pdf).

First, we compute topographic potential (LS factor) for soil erosion and compare impact of the power function exponents on the erosion pattern.

In [ ]:
gs.run_command("g.region", raster="elev_lid792_1m", flags="p")
gs.run_command("r.slope.aspect", elevation="elev_lid792_1m", slope="slope_1m", aspect="aspect_1m")
gs.run_command("r.flow", elevation="elev_lid792_1m", flowaccumulation="flowaccum_1m")
gs.run_command(
    "r.mapcalc",
    expression="lsfac3d_1m = 1.2 * pow(flowaccum_1m * 1./22.1,0.2) * pow(sin(slope_1m)/0.09,1.3)",
)
gs.run_command(
    "r.mapcalc",
    expression="lsfac3d_s1_1m = 1.5 * pow(flowaccum_1m * 1./22.1,0.5) * pow(sin(slope_1m)/0.09,1.0)",
)
gs.run_command("r.colors", map="lsfac3d_s1_1m", color="gyr", flags="e")
gs.run_command("r.colors", map="lsfac3d_1m", raster="lsfac3d_s1_1m")

Display the LS factor maps with contours:

In [ ]:
lsfac_s13_map = gj.Map(filename="lsfac_s13.png")
lsfac_s13_map.d_rast(map="lsfac3d_1m")
lsfac_s13_map.d_vect(map="elev_lid792_cont1m")
lsfac_s13_map.d_legend(raster="lsfac3d_1m", at=(2, 30, 2, 6), flags="d")
lsfac_s13_map.d_barscale()
lsfac_s13_map.show()

In [ ]:
lsfac_s10_map = gj.Map(filename="lsfac_s10.png")
lsfac_s10_map.d_rast(map="lsfac3d_s1_1m")
lsfac_s10_map.d_vect(map="elev_lid792_cont1m")
lsfac_s10_map.d_legend(raster="lsfac3d_s1_1m", at=(2, 30, 2, 6), flags="d")
lsfac_s10_map.show()

#### Question 2.1

Which equation represents conditions when contributing area has greater impact and which models stronger influence of slope? Which result predicts higher erosion rates?

`Add text here with your answer to question 2.1`

---
## Part 3: Compute soil detachment for spatially variable land cover

Recode `landcover_1m` to land cover erosion factor C values given in the [cfac_rules.txt](data/cfac_rules.txt) table using `r.recode`. Assign a custom color table and display result.

In [ ]:
gs.run_command("r.recode", input="landcover_1m", output="cfactor_1m", rules="cfac_rules.txt")
gs.run_command("r.colors", map="cfactor_1m", rules="cfac_color.txt")

In [ ]:
cfactor_map = gj.Map(filename="cfactor_bare.png")
cfactor_map.d_rast(map="cfactor_1m")
cfactor_map.d_legend(raster="cfactor_1m", at=(2, 40, 2, 6))
cfactor_map.show()

Compute the USLE3D equation using map algebra. `cfactorbare_1m` is the same as `cfactor_1m` with bare agricultural fields, `cfactorgrow_1m` has landuse recoded for growing season conditions. We use rainfall intensity factor R=270 and soil erodibility factor K extracted from the soils database.

Assign custom color ramp and specify units of the raster maps.

In [ ]:
gs.run_command("r.mapcalc", expression="soillossbare_1m = 270. * soils_Kfactor * lsfac3d_1m * cfactorbare_1m")
gs.run_command("r.mapcalc", expression="soillossgrow_1m = 270. * soils_Kfactor * lsfac3d_1m * cfactorgrow_1m")
gs.run_command("r.colors", map="soillossbare_1m", rules="soilloss_color.txt")
gs.run_command("r.colors", map="soillossgrow_1m", raster="soillossbare_1m")
gs.run_command("r.support", map="soillossbare_1m", units="ton/(acre.year)")
gs.run_command("r.support", map="soillossgrow_1m", units="ton/(acre.year)")

Display the soil loss maps for two different seasons:

In [ ]:
soillossbare_map = gj.Map(filename="soillossbare.png")
soillossbare_map.d_rast(map="soillossbare_1m")
soillossbare_map.d_legend(raster="soillossbare_1m", at=(2, 40, 2, 6))
soillossbare_map.show()

In [ ]:
soillossgrow_map = gj.Map(filename="soillossgrow.png")
soillossgrow_map.d_rast(map="soillossgrow_1m")
soillossgrow_map.d_legend(raster="soillossgrow_1m", at=(2, 40, 2, 6))
soillossgrow_map.show()

Compute summary statistics for erosion rates:

In [ ]:
gs.parse_command("r.univar", map="soillossbare_1m")
gs.parse_command("r.univar", map="soillossgrow_1m")

#### Question 3.1

Compare erosion rates and distribution for bare and growing season conditions.

`Add text here with your answer to question 3.1`

---
## Part 4: Compute net erosion/deposition using USPED

Compute net erosion/deposition maps as divergence of sediment flow (USPED).

First compute sediment flow and its x, y components in the steepest slope (aspect) direction:

In [ ]:
gs.run_command(
    "r.mapcalc",
    expression="sedflow_1m = 270. * soils_Kfactor * cfactorgrow_1m * flowaccum_1m * sin(slope_1m)",
)
gs.run_command("r.colors", map="sedflow_1m", raster="soillossbare_1m")

In [ ]:
sedflow_map = gj.Map(filename="sedflow.png")
sedflow_map.d_rast(map="sedflow_1m")
sedflow_map.d_legend(raster="sedflow_1m", at=(2, 40, 2, 6))
sedflow_map.show()

Compute the x and y components of sediment flow:

In [ ]:
gs.run_command("r.mapcalc", expression="qsx = sedflow_1m * cos(aspect_1m)")
gs.run_command("r.mapcalc", expression="qsy = sedflow_1m * sin(aspect_1m)")

Compute change of sediment flow in the x and y directions as partial derivatives of sediment flow component surfaces. We use `qsx` and `qsy` surfaces (rasters) instead of elevation raster as input and partial derivatives `dx` and `dy` as output of `r.slope.aspect`.

The change of sediment flow in the direction of flow (aspect) is then computed as divergence resulting in net erosion and deposition map `erdep`.

In [ ]:
gs.run_command("r.slope.aspect", elevation="qsx", dx="qsx_dx")
gs.run_command("r.slope.aspect", elevation="qsy", dy="qsy_dy")
gs.run_command("r.mapcalc", expression="erdep = qsx_dx + qsy_dy")
gs.parse_command("r.info", map="erdep", flags="r")
gs.run_command("r.colors", map="erdep", rules="erdep_color.txt")

Display the map of erosion and deposition pattern:

In [ ]:
erdep_map = gj.Map(filename="erosion_deposition.png")
erdep_map.d_rast(map="erdep")
erdep_map.d_legend(raster="erdep", at=(2, 50, 2, 6), range=(-5, 5))
erdep_map.show()

Optional: Display `elev_lid792_1m` in 3D and drape over `erdep` as color.

### Compute summary statistics

Use `r.recode` to classify erosion/deposition and `r.category` to add labels (stable, high erosion, etc) to individual categories:

In [ ]:
gs.run_command("r.recode", input="erdep", output="erdep_class", rules="erdep_class.txt")
gs.run_command("r.category", map="erdep_class", rules="erdep_label.txt", separator=":")
gs.run_command("r.colors", map="erdep_class", color="differences", flags="n")

In [ ]:
gs.parse_command("r.report", map="erdep_class", unit="p,h,a")

In [ ]:
erdep_class_map = gj.Map(filename="erdep_classes.png")
erdep_class_map.d_rast(map="erdep_class")
erdep_class_map.d_legend(raster="erdep_class", at=(60, 80, 2, 6), flags="c")
erdep_class_map.show()

#### Question 4.1

Comment on advantages, disadvantages and risks of classifying erosion/deposition data into categories.

`Add text here with your answer to question 4.1`

---
## Part 5: Optional - Sediment transport and eroded DEM

### Estimate amount of excess sediment transported out of the studied site

Compute univariate statistics for the `erdep` map and extract the sum:

In [ ]:
gs.parse_command("r.univar", map="erdep")

The units are tons/(acre.year), but our cells are 1 m<sup>2</sup>. Therefore we have to divide by 4046 (1 acre = 4046 m<sup>2</sup>) to get total ton per year transported from the watershed.

In [ ]:
# Divide the sum by 4046 to convert to tons/year
univar = gs.parse_command("r.univar", map="erdep", flags="g")
total_sum = float(univar["sum"])
print(f"Sum: {total_sum:.6f} ton/(acre.year)")
print(f"Total sediment transported: {total_sum / 4046:.6f} ton/year")

### Compute new DEM with erosion carved-in

In [ ]:
gs.run_command("r.mapcalc", expression="elev_erodedb_1m = elev_lid792_1m-(soillossbare_1m/100.)")

Display `elev_erodedb_1m` in 3D and drape over `soillossbare_1m` as color. To view it in 3D switch off everything except `elev_erodedb_1m`. Drape `soillossbare_1m` as color and set fine resolution to 1. Use lighting from SW, z-exag at least 2.0.

In [ ]:
eroded_3d_map = gj.Map3D()
eroded_3d_map.render(
    elevation_map="elev_erodedb_1m",
    color_map="soillossbare_1m",
    perspective=20,
    zexag=2.0,
)
eroded_3d_map.overlay.d_legend(raster="soillossbare_1m", at=(60, 97, 87, 92))
eroded_3d_map.show()